# Buổi 18 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `hoi_quy.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Hồi quy giả (mục 4.2)

In [ ]:
%matplotlib inline
import warnings

import hoi_quy as hq
import pandas as pd

warnings.simplefilter("ignore")
y, X = hq.cap_gia()
print({k: round(v, 4) for k, v in hq.hoi_quy_ols(y, X).items()})
print("mô phỏng 1.000 cặp bước ngẫu nhiên độc lập:", hq.mo_phong_hoi_quy_gia(1000))

## Bước 2 — Hồi quy động (mục 4.3)

Sửa `he_so_ip` rồi chạy lại ô này.

In [ ]:
print({k: round(v, 4) for k, v in hq.he_so_ip(y, X).items()})

## Bước 3 — Tải ERCOT: Fourier, nhiệt độ, chọn K (mục 4.4)

In [ ]:
from statsforecast.models import AutoARIMA

dien, nhiet = hq.doc_dien(), hq.doc_nhiet()
df = pd.DataFrame({"unique_id": "ERCO", "ds": dien.index, "y": dien.to_numpy()})
cutoff = pd.Timestamp("2024-08-09 06:00")
lich_su = df[(df["ds"] <= cutoff) & (df["ds"] > cutoff - pd.Timedelta(hours=hq.HOC))]
print(hq.bien_giai_thich(pd.DatetimeIndex(lich_su["ds"][:3]), nhiet["that"], 1, 1).round(2).to_string())
print(hq.chon_K(lich_su, nhiet=nhiet).round(1).to_string(index=False))
X_hoc = hq.bien_giai_thich(pd.DatetimeIndex(lich_su["ds"]), nhiet["that"])
mh = AutoARIMA(season_length=1).fit(lich_su["y"].to_numpy(), X=X_hoc[hq._cot_dung(X_hoc)].to_numpy())
print("sai số ARIMA (p, q, P, Q, m, d, D):", mh.model_["arma"], "| Ljung–Box p:", hq.ljung_box_p(mh.model_["residuals"]))

## Bước 4 — So bốn cách đa mùa vụ (mục 4.4)

Lần đầu khoảng 7 phút; kết quả lưu vào `lab/du-lieu/cache/`.

In [ ]:
from tv.backtest import diebold_mariano

kq = hq.backtest_dien(luu=True)
print("thứ của các cutoff:", sorted(kq["cutoff"].dt.day_name().unique()))
mae = hq.mae_theo_cua_so(kq)
print(pd.DataFrame({"MAE TB": mae.mean(), "MAE trung vị": mae.median()}).round(0).sort_values("MAE TB").to_string())
for khac in ("SN24", "MSTL", "OLS", "Prophet"):
    dm = diebold_mariano((kq["y"] - kq["DHR"]).to_numpy(), (kq["y"] - kq[khac]).to_numpy(), h=24, ham_mat_mat="tuyet_doi")
    print(f"DHR so với {khac}: DM {dm.thong_ke:.2f}, p {dm.p_value:.3f}")

## Bước 5 — Prophet và Tết (mục 4.6)

Đổi `KHAI_TET = True` trong `hoi_quy.py` rồi chạy lại ô này.

In [ ]:
kq_w = hq.backtest_wiki()
for nam in (2024, 2025):
    k = kq_w[kq_w["ds"].dt.year == nam]
    print(nam, "| KHAI_TET =", hq.KHAI_TET,
          "| MAE quanh Tết", round(hq.mae_quanh_tet(k), 3),
          "| MAE cả năm: Prophet", round(float((k["y"] - k["Prophet"]).abs().mean()), 3),
          "| seasonal naive 364", round(float((k["y"] - k["SN364"]).abs().mean()), 3))

Trong terminal ở thư mục `lab/`: `python lab.py check` — phải xanh 5/5.